# Zomato Bangalore — Exploratory Data Analysis

**Dataset:** Kaggle [Zomato Bangalore Restaurants](https://www.kaggle.com/datasets/himanshupoddar/zomato-bangalore-restaurants)  
**Author:** Meet Modi  
**Tools:** Python · Pandas · Plotly

## 0. Setup & Data Loading

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
import os

# Paths
DATA_PATH = Path("../data/processed/zomato_clean.csv")
PLOTS_DIR = Path("../plots")
PLOTS_DIR.mkdir(exist_ok=True)

# Plotly defaults
TEMPLATE = "plotly_dark"
COLOR_SEQ = px.colors.sequential.Tealgrn

df = pd.read_csv(DATA_PATH)
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

## 1. Data Overview

In [ ]:
print("Shape:", df.shape)
print("\nColumn dtypes:")
print(df.dtypes)
print("\nNull counts:")
print(df.isnull().sum())
print("\nSample rows:")
df.head()

In [ ]:
df.describe()

## 2. Rating Distribution

In [ ]:
# Histogram of ratings
rated = df.dropna(subset=["rate"])

fig = px.histogram(
    rated, x="rate", nbins=18,
    title="Distribution of Restaurant Ratings",
    labels={"rate": "Rating (out of 5)", "count": "Number of Restaurants"},
    color_discrete_sequence=["#00b4d8"],
    template=TEMPLATE,
)
fig.update_layout(bargap=0.05, yaxis_title="Number of Restaurants")
fig.write_image(PLOTS_DIR / "rating_distribution.png", width=900, height=500, scale=2)
fig.show()

In [ ]:
# Average rating by location (top 15)
loc_rating = (
    rated.groupby("location")["rate"]
    .agg(["mean", "count"])
    .query("count >= 30")
    .sort_values("mean", ascending=False)
    .head(15)
    .reset_index()
)
loc_rating.columns = ["location", "avg_rating", "count"]

fig = px.bar(
    loc_rating, x="avg_rating", y="location",
    orientation="h",
    title="Top 15 Locations by Average Rating (min 30 restaurants)",
    labels={"avg_rating": "Average Rating", "location": ""},
    color="avg_rating",
    color_continuous_scale="Tealgrn",
    template=TEMPLATE,
)
fig.update_layout(yaxis={"categoryorder": "total ascending"}, showlegend=False)
fig.write_image(PLOTS_DIR / "avg_rating_by_location.png", width=900, height=500, scale=2)
fig.show()

## 3. Cost Analysis

In [ ]:
# Distribution of approx_cost
cost_df = df.dropna(subset=["approx_cost"])

fig = px.histogram(
    cost_df, x="approx_cost", nbins=50,
    title="Distribution of Approximate Cost for Two",
    labels={"approx_cost": "Cost for Two (₹)", "count": "Restaurants"},
    color_discrete_sequence=["#e76f51"],
    template=TEMPLATE,
)
fig.update_layout(bargap=0.05, yaxis_title="Number of Restaurants")
fig.write_image(PLOTS_DIR / "cost_distribution.png", width=900, height=500, scale=2)
fig.show()

In [ ]:
# Cost vs Rating scatter
both = df.dropna(subset=["rate", "approx_cost"])

fig = px.scatter(
    both, x="approx_cost", y="rate",
    title="Cost vs Rating",
    labels={"approx_cost": "Cost for Two (₹)", "rate": "Rating"},
    opacity=0.3,
    color_discrete_sequence=["#2a9d8f"],
    trendline="ols",
    template=TEMPLATE,
)
fig.write_image(PLOTS_DIR / "cost_vs_rating.png", width=900, height=500, scale=2)
fig.show()

## 4. Online Order & Table Booking

In [ ]:
# Online ordering stats
online_pct = df["online_order"].mean() * 100
print(f"Restaurants with online ordering: {online_pct:.1f}%")

# Average rating: online vs offline
online_rating = rated.groupby("online_order")["rate"].mean().reset_index()
online_rating["online_order"] = online_rating["online_order"].map({True: "Online", False: "Offline"})

fig = px.bar(
    online_rating, x="online_order", y="rate",
    title="Average Rating: Online vs Offline Ordering",
    labels={"online_order": "", "rate": "Average Rating"},
    color="online_order",
    color_discrete_map={"Online": "#00b4d8", "Offline": "#e76f51"},
    template=TEMPLATE,
)
fig.update_layout(showlegend=False)
fig.write_image(PLOTS_DIR / "online_vs_offline_rating.png", width=700, height=450, scale=2)
fig.show()

In [ ]:
# Table booking stats
booking_pct = df["book_table"].mean() * 100
print(f"Restaurants with table booking: {booking_pct:.1f}%")

booking_rating = rated.groupby("book_table")["rate"].mean().reset_index()
booking_rating["book_table"] = booking_rating["book_table"].map({True: "Booking Available", False: "No Booking"})

fig = px.bar(
    booking_rating, x="book_table", y="rate",
    title="Average Rating: Table Booking vs No Booking",
    labels={"book_table": "", "rate": "Average Rating"},
    color="book_table",
    color_discrete_map={"Booking Available": "#2a9d8f", "No Booking": "#f4a261"},
    template=TEMPLATE,
)
fig.update_layout(showlegend=False)
fig.write_image(PLOTS_DIR / "booking_vs_no_booking.png", width=700, height=450, scale=2)
fig.show()

## 5. Location Analysis

In [ ]:
# Top 15 locations by restaurant count
top_locs = df["location"].value_counts().head(15).reset_index()
top_locs.columns = ["location", "count"]

# Merge avg rating per location
avg_rate_loc = rated.groupby("location")["rate"].mean().reset_index()
avg_rate_loc.columns = ["location", "avg_rating"]
top_locs = top_locs.merge(avg_rate_loc, on="location", how="left")

fig = px.bar(
    top_locs, x="count", y="location",
    orientation="h",
    title="Top 15 Locations by Restaurant Count",
    labels={"count": "Number of Restaurants", "location": ""},
    color="avg_rating",
    color_continuous_scale="Tealgrn",
    template=TEMPLATE,
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.write_image(PLOTS_DIR / "top_locations_count.png", width=900, height=550, scale=2)
fig.show()

In [ ]:
# Top 10 locations by avg rating (min 30 restaurants)
loc_agg = (
    rated.groupby("location")["rate"]
    .agg(["mean", "count"])
    .query("count >= 30")
    .sort_values("mean", ascending=False)
    .head(10)
    .reset_index()
)
loc_agg.columns = ["location", "avg_rating", "count"]

fig = px.bar(
    loc_agg, x="avg_rating", y="location",
    orientation="h",
    title="Top 10 Locations by Average Rating (min 30 restaurants)",
    labels={"avg_rating": "Average Rating", "location": ""},
    color="avg_rating",
    color_continuous_scale="Viridis",
    template=TEMPLATE,
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.write_image(PLOTS_DIR / "top_locations_rating.png", width=900, height=450, scale=2)
fig.show()

## 6. Cuisine Analysis

In [ ]:
# Explode cuisines
cuisine_series = (
    df.dropna(subset=["cuisines"])["cuisines"]
    .str.split(",")
    .explode()
    .str.strip()
)

# Top 20 most common cuisines
top_cuisines = cuisine_series.value_counts().head(20).reset_index()
top_cuisines.columns = ["cuisine", "count"]

fig = px.bar(
    top_cuisines, x="count", y="cuisine",
    orientation="h",
    title="Top 20 Most Common Cuisines",
    labels={"count": "Number of Restaurants", "cuisine": ""},
    color="count",
    color_continuous_scale="Sunset",
    template=TEMPLATE,
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.write_image(PLOTS_DIR / "top_cuisines_count.png", width=900, height=600, scale=2)
fig.show()

In [ ]:
# Highest-rated cuisines (min 50 restaurants)
cuisine_df = df.dropna(subset=["cuisines", "rate"]).copy()
cuisine_exploded = cuisine_df.assign(
    cuisine=cuisine_df["cuisines"].str.split(",")
).explode("cuisine")
cuisine_exploded["cuisine"] = cuisine_exploded["cuisine"].str.strip()

cuisine_agg = (
    cuisine_exploded.groupby("cuisine")["rate"]
    .agg(["mean", "count"])
    .query("count >= 50")
    .sort_values("mean", ascending=False)
    .head(15)
    .reset_index()
)
cuisine_agg.columns = ["cuisine", "avg_rating", "count"]

fig = px.bar(
    cuisine_agg, x="avg_rating", y="cuisine",
    orientation="h",
    title="Highest-Rated Cuisines (min 50 restaurants)",
    labels={"avg_rating": "Average Rating", "cuisine": ""},
    color="avg_rating",
    color_continuous_scale="Tealgrn",
    template=TEMPLATE,
    text="count",
)
fig.update_traces(texttemplate="%{text} restaurants", textposition="inside")
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.write_image(PLOTS_DIR / "top_cuisines_rating.png", width=900, height=500, scale=2)
fig.show()

## 7. Restaurant Type Analysis

In [ ]:
# Breakdown by restaurant type
type_df = (
    df.dropna(subset=["restaurant_type"])
    ["restaurant_type"].value_counts().head(10).reset_index()
)
type_df.columns = ["restaurant_type", "count"]

fig = px.bar(
    type_df, x="count", y="restaurant_type",
    orientation="h",
    title="Top 10 Restaurant Types",
    labels={"count": "Number of Restaurants", "restaurant_type": ""},
    color="count",
    color_continuous_scale="Blugrn",
    template=TEMPLATE,
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.write_image(PLOTS_DIR / "restaurant_type_breakdown.png", width=900, height=450, scale=2)
fig.show()

In [ ]:
# Average cost per restaurant type
type_cost = (
    df.dropna(subset=["restaurant_type", "approx_cost"])
    .groupby("restaurant_type")["approx_cost"]
    .agg(["mean", "count"])
    .query("count >= 20")
    .sort_values("mean", ascending=False)
    .head(10)
    .reset_index()
)
type_cost.columns = ["restaurant_type", "avg_cost", "count"]

fig = px.bar(
    type_cost, x="avg_cost", y="restaurant_type",
    orientation="h",
    title="Average Cost for Two by Restaurant Type",
    labels={"avg_cost": "Avg Cost for Two (₹)", "restaurant_type": ""},
    color="avg_cost",
    color_continuous_scale="Oryel",
    template=TEMPLATE,
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.write_image(PLOTS_DIR / "avg_cost_per_type.png", width=900, height=450, scale=2)
fig.show()

## 8. Key Business Insights

In [ ]:
# --- Compute key numbers for insights ---
total = len(df)
rated_count = len(rated)
avg_rating = rated["rate"].mean()
median_cost = df["approx_cost"].median()

# Online order metrics
online_avg = rated[rated["online_order"] == True]["rate"].mean()
offline_avg = rated[rated["online_order"] == False]["rate"].mean()

# Table booking metrics
booking_avg = rated[rated["book_table"] == True]["rate"].mean()
no_booking_avg = rated[rated["book_table"] == False]["rate"].mean()

# Top location
top_loc = df["location"].value_counts().index[0]
top_loc_count = df["location"].value_counts().iloc[0]

# Top cuisine
top_cuisine = cuisine_series.value_counts().index[0]

print("="*60)
print("  KEY BUSINESS INSIGHTS")
print("="*60)
print(f"""
1. ONLINE ORDERING DRIVES VOLUME, NOT QUALITY
   {online_pct:.0f}% of restaurants accept online orders, yet their avg
   rating ({online_avg:.2f}) is slightly {'higher' if online_avg > offline_avg else 'lower'} than offline ({offline_avg:.2f}).
   → For new restaurants, enabling online ordering boosts
     discoverability without sacrificing perceived quality.

2. TABLE BOOKING CORRELATES WITH PREMIUM QUALITY
   Only {booking_pct:.0f}% of restaurants offer table booking, but they
   average {booking_avg:.2f} vs {no_booking_avg:.2f} for non-booking — a {(booking_avg - no_booking_avg):.2f}-point gap.
   → Restaurants offering bookings likely invest more in experience,
     which is reflected in higher ratings.

3. LOCATION SATURATION: {top_loc.upper()}
   {top_loc} leads with {top_loc_count:,} restaurants — extreme competition.
   New entrants should consider less-saturated locations with
   comparable ratings for better unit economics.

4. COST-QUALITY SWEET SPOT
   Median cost is ₹{median_cost:,.0f} for two. The cost-vs-rating
   scatter shows diminishing returns above ~₹800.
   → A restaurant priced at ₹400–600 can achieve 3.8+ ratings,
     making mid-range pricing optimal for new entrants.

5. CUISINE: {top_cuisine.upper()} DOMINATES
   {top_cuisine} is the most listed cuisine but not the highest-rated.
   Niche cuisines (e.g., European, Hyderabadi) with 50+ listings
   score higher avg ratings — suggesting underserved demand
   for specialty food in Bangalore.
""")
print("="*60)